In [0]:
from pyspark.sql import functions as F

orders_path = (
    "/Volumes/workspace/olist_landing/batched_files/"
    "batch_streaming/orders/*.csv"
)

synthetic_order_ids = (
    spark.read.option("header", "true").csv(orders_path)
    .select("order_id")
    .distinct()
)

reviewed_order_ids = (
    spark.table("workspace.olist_silver.customer_reviews")
    .select("order_id")
    .distinct()
)

eligible_orders = (
    spark.table("workspace.olist_silver.orders_customer")
    .join(synthetic_order_ids, "order_id", "left_semi")
    .join(reviewed_order_ids, "order_id", "left_anti")
    .filter(F.col("order_status") == "delivered")
    .withColumn(
        "simulated_delivery_at",
        F.col("order_delivered_customer_date"),
    )
    .filter(F.col("simulated_delivery_at") <= F.current_timestamp())
    .select("order_id", "simulated_delivery_at")
    .dropDuplicates()
)

display(eligible_orders.orderBy("simulated_delivery_at"))

In [0]:
sample_orders = (
    eligible_orders
    .groupBy("order_id")
    .agg(F.max("simulated_delivery_at").alias("simulated_delivery_at"))
    .orderBy("simulated_delivery_at", "order_id")
)

reviews_preview = (
    sample_orders
    .withColumn("scenario", F.pmod(F.xxhash64("order_id"), F.lit(4)))
    .select(
        F.md5(F.concat(F.lit("review-sintetica:"), F.col("order_id"))).alias("review_id"),
        "order_id",
        F.expr("element_at(array(5, 1, 3, 4), CAST(scenario AS INT) + 1)").alias("review_score"),
        F.lit(None).cast("string").alias("review_comment_title"),
        F.expr("""
            element_at(
                array(
                    'Produto excelente, recomendo!',
                    'O produto chegou com defeito e quero trocar.',
                    'Recebi o pedido.',
                    'Gostei do produto, mas a embalagem chegou amassada.'
                ),
                CAST(scenario AS INT) + 1
            )
        """).alias("review_comment_message"),
        F.col("simulated_delivery_at").alias("review_creation_date"),
        F.expr("simulated_delivery_at + INTERVAL 1 HOUR").alias("review_answer_timestamp"),
        F.lit("synthetic").alias("review_source"),
    )
    .filter(F.col("review_answer_timestamp") <= F.current_timestamp())
)

display(reviews_preview)

In [0]:
comments_by_score = {
    5: [
        "Produto excelente, recomendo!",
        "Adorei a compra. Veio tudo certinho!",
        "Chegou antes do prazo e bem embalado.",
        "Qualidade melhor do que eu esperava.",
        "Preço ótimo e compra supertranquila.",
    ],
    1: [
        "O produto chegou com defeito e quero trocar.",
        "Veio quebrado e não consegui usar.",
        "Recebi um item diferente do que comprei.",
        "A entrega demorou muito além do prazo informado.",
        "Pedi ajuda para trocar o produto e fiquei sem resposta.",
    ],
    3: [
        "Recebi o pedido.",
        "O pacote chegou ontem à tarde.",
        "O produto foi entregue na portaria.",
        "Comprei para dar de presente.",
        "Ainda não testei o produto.",
    ],
    4: [
        "Gostei do produto, mas a embalagem chegou amassada.",
        "O preço foi bom, mas a entrega demorou mais do que eu esperava.",
        "Chegou rápido, mas a cor é um pouco diferente da foto.",
        "O produto funciona bem, mas o manual é confuso.",
        "O atendimento foi atencioso, mas a troca demorou.",
    ],
}

comment_groups = []
for score, comments in comments_by_score.items():
    comment_groups.extend([
        F.lit(score),
        F.array(*[F.lit(comment) for comment in comments]),
    ])

reviews_preview = (
    reviews_preview
    .withColumn(
        "comment_options",
        F.element_at(F.create_map(*comment_groups), F.col("review_score")),
    )
    .withColumn(
        "review_comment_message",
        F.expr("""
            element_at(
                comment_options,
                CAST(
                    pmod(xxhash64(order_id, 'review_text'), size(comment_options))
                    AS INT
                ) + 1
            )
        """),
    )
    .drop("comment_options")
)

display(reviews_preview.select("order_id", "review_score", "review_comment_message"))

In [0]:
from delta.tables import DeltaTable

target_table = "workspace.olist_silver.customer_reviews"
(
    DeltaTable.forName(spark, target_table)
    .alias("target")
    .merge(
        reviews_preview.alias("updates"),
        "target.order_id = updates.order_id",
    )
    .whenNotMatchedInsertAll()
    .execute()

)


In [0]:
%sql
DESCRIBE HISTORY workspace.olist_silver.customer_reviews;

In [0]:
%sql
select distinct review_source, count(*)
from workspace.olist_silver.customer_reviews
group by review_source



In [0]:
%sql
select distinct o.order_status--, r.review_comment_message 
from workspace.olist_silver.customer_reviews r
inner join workspace.olist_silver.orders_customer o
on r.order_id = o.order_id
where r.review_source = 'synthetic'


In [0]:
%sql
select distinct order_status from workspace.olist_silver.orders_customer o
where order_status = ''